In [1]:
pip install streamlit google-generativeai pymysql sqlalchemy pandas

  Using cached uritemplate-4.2.0-py3-none-any.whl.metadata (2.6 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.3 MB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 3.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/15.6 MB ? eta -:--:--
   -- ------------------------------------- 1.0/15.6 MB 4.2 MB/s eta 0:00:04
   ---- --------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.


In [63]:
import pandas as pd
from groq import Groq
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# ==========================================
# CONFIGURATION
# ==========================================

GROQ_API_KEY = "YOUR_GROQ_API_KEY"

client = Groq(api_key=GROQ_API_KEY)


In [65]:
# ==========================================
# MYSQL CONNECTION
# ==========================================

url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password="Hari@0805",
    host="127.0.0.1",
    port=3306,
    database="walmartsales"
)

engine = create_engine(url)

In [67]:
# ==========================================
# TEST DATABASE CONNECTION
# ==========================================

try:
    with engine.connect() as conn:
        print(" Connected to MySQL Successfully!\n")
except Exception as e:
    print(" Database Connection Failed")
    print(e)
    exit()

 Connected to MySQL Successfully!



In [69]:
# ==========================================
# USER QUESTION
# ==========================================

question = input("Ask a business question: ")

# ==========================================
# PROMPT FOR SQL GENERATION
# ==========================================

prompt = f"""
You are an expert MySQL developer.

Database Name:
walmartsales

Table Name:
walmart_sales

Columns:
Store
Dept
Date
Weekly_Sales
IsHoliday
Temperature
Fuel_Price
MarkDown1
MarkDown2
MarkDown3
MarkDown4
MarkDown5
CPI
Unemployment
Type
Size
Year
Month
Month_Name
Quarter
Week

Rules:
1. Return ONLY SQL.
2. Do NOT explain.
3. Do NOT use markdown.
4. MySQL syntax only.

Question:

{question}
"""

Ask a business question:  no. of departments


In [71]:
# ==========================================
# GENERATE SQL
# ==========================================

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

sql = response.choices[0].message.content

sql = (
    sql.replace("```sql", "")
       .replace("```", "")
       .strip()
)

print("\n==============================")
print("GENERATED SQL")
print("==============================\n")

print(sql)


GENERATED SQL

SELECT COUNT(DISTINCT Dept) FROM walmart_sales


In [75]:
# ==========================================
# EXECUTE SQL
# ==========================================

try:

    df = pd.read_sql(text(sql), engine)

    print("\n==============================")
    print("QUERY RESULT")
    print("==============================\n")

    print(df)

    # ==========================================
    # AI BUSINESS INSIGHTS
    # ==========================================

    insight_prompt = f"""
You are a Senior Business Analyst.

User Question:
{question}

Generated SQL:
{sql}

Query Result:

{df.to_string(index=False)}

Analyze the result.

Provide:

1. Key Findings
2. Business Insights
3. Recommendations

Keep the answer short and professional.
"""

    insight = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": insight_prompt
            }
        ]
    )

    print("\n==============================")
    print("AI BUSINESS INSIGHTS")
    print("==============================\n")

    print(insight.choices[0].message.content)

except Exception as e:

    print("\nSQL Execution Error")
    print(e)


QUERY RESULT

   COUNT(DISTINCT Dept)
0                    81

AI BUSINESS INSIGHTS

**Analysis of Department Count**

Based on the query result, here are the key findings, business insights, and recommendations:

1. **Key Findings**: There are 81 unique departments in the Walmart sales data.
2. **Business Insights**: The large number of departments suggests a diverse product range, which can be beneficial for customer choice but also poses a challenge for inventory management and sales optimization.
3. **Recommendations**: Consider department-level analysis to identify top-performing departments, optimize inventory allocation, and develop targeted marketing strategies to improve overall sales performance.
